# Technical Support Agent
**ResolveX | M5: Technical Support Agent**

This notebook evaluates how the current support agent combines ticket classification, evidence retrieval, response generation, and safety checks.

## Evaluation cases

One held-out Ask Ubuntu ticket and three synthetic tickets exercise four selected paths: kernel-removal escalation, a read-only DNS recommendation, missing repair details, and an out-of-scope request.

The synthetic tickets demonstrate selected behaviors; they are not test-set evidence. The M4 community index represents a 5,000-case comparison sample rather than the full 240,555-case dataset, so retrieval coverage is limited.

Run the notebook from top to bottom. The held-out kernel-removal case and DNS case call Gemini; missing-details and out-of-scope cases return before generation. A Gemini availability error (such as 503) means that generation was not evaluated for that case. After changing source code, restart the kernel and rerun the notebook so the results come from one code version.

## 1. Setup

Locate the project and check that the prepared data and M3/M4 artifacts are available.

In [1]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import sys
import csv

cwd = Path.cwd().resolve()
ROOT = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "src/support_agent.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the ResolveX workspace.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT_ID = "M5-SELECTED-CASES"
DATASET_PATH = ROOT / "data/processed/ml_dataset.csv"
SPLITS_PATH = ROOT / "data/processed/classification_splits.csv"
CLASSIFIER_PATH = ROOT / "data/processed/classifier.joblib"
RETRIEVAL_ROOT = ROOT / "data/processed/retrieval"
RUN_RECORDS_ROOT = ROOT / "data/processed/m5_evaluation_runs"
GENERATION_ATTEMPTED = False
ADDITIONAL_CASES_ATTEMPTED = False

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"

print("Workspace:", ROOT)
print("Experiment:", EXPERIMENT_ID)
print("Python:", sys.version.split()[0])
print("scikit-learn:", package_version("scikit-learn"))
print("pandas:", package_version("pandas"))
print("Test dataset exists:", DATASET_PATH.is_file())
print("Split assignments exist:", SPLITS_PATH.is_file())
print("Saved M3 classifier exists:", CLASSIFIER_PATH.is_file())
print("Saved M4 retrieval artifacts exist:", RETRIEVAL_ROOT.is_dir())
run_record_count = (
    sum(1 for item in RUN_RECORDS_ROOT.rglob("*") if item.is_file())
    if RUN_RECORDS_ROOT.is_dir() else 0
)
print("Saved evaluation record files:", run_record_count)

def read_csv_row(path, id_column, wanted_id):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            if row.get(id_column) == wanted_id:
                return row
    return None


Workspace: C:\Users\jc\Desktop\ResolveX
Experiment: M5-SELECTED-CASES
Python: 3.14.7
scikit-learn: 1.9.0
pandas: 2.3.3
Test dataset exists: True
Split assignments exist: True
Saved M3 classifier exists: True
Saved M4 retrieval artifacts exist: True
Saved evaluation record files: 0


## 2. Load the M3/M4 Artifacts and Agent

Load the saved classifier and retrieval indexes, then configure the Gemini chain used by the cases below. This setup does not send a generation request.

In [2]:
import os
import joblib
from dotenv import load_dotenv
from src.retrieval import load_retriever
from src.official_docs import load_official_docs
from src.support_agent import SupportAgent, make_m5_chains
from src.llm_provider import configured_model_name

classifier_artifact = joblib.load(CLASSIFIER_PATH)
retriever = load_retriever(ROOT)
if retriever.dense is None:
    raise RuntimeError("The saved M4 retriever has no dense model to share with the docs index.")
official_retriever, official_manifest = load_official_docs(ROOT, retriever.dense.model)

load_dotenv(ROOT / ".env")
api_key = os.getenv("GEMINI_API_KEY", "").strip()
if not api_key:
    raise ValueError("GEMINI_API_KEY is missing; no Gemini request was made.")

agent = SupportAgent(
    classifier=classifier_artifact,
    retriever=retriever,
    chain=make_m5_chains(api_key=api_key, provider="gemini")["chain"],
    official_retriever=official_retriever,
)
print("Classifier and retrieval artifacts loaded.")
print("Configured model:", configured_model_name("gemini"))
print("Official documentation pages:", official_manifest["source_count"])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Classifier and retrieval artifacts loaded.
Configured model: gemini-3.5-flash-lite
Official documentation pages: 16


## 3. Inspect Components, Then Run the Integrated Agent

For the held-out kernel case and the synthetic DNS case, inspect M3 classification, M4 community retrieval, and official-document retrieval separately. Then `prepare()` runs the integrated pipeline and `generate()` produces the agent result. This helps distinguish component evidence from the final decision.

The missing-details and out-of-scope cases exercise early decision gates and should return before Gemini is called. The kernel-removal and DNS cases may call Gemini. If generation is unavailable, the case remains visible with its error; that is not evidence of answer quality.

In [3]:
import re

expected_ids = ["1503371", "SYN-DNS-001", "SYN-APT-MISSING-001", "SYN-OUT-OF-SCOPE-001"]
# Keep only results for the cases retained in this notebook.
current_run_results = [
    item for item in globals().get("current_run_results", [])
    if item.get("case_id") in expected_ids
]

second_case_id = "1503371"
second_row = read_csv_row(DATASET_PATH, "Id", second_case_id)
second_split = read_csv_row(SPLITS_PATH, "Id", second_case_id)
if second_row is None:
    raise KeyError(f"Question {second_case_id} is not in the prepared dataset.")
if second_split is None or second_split.get("Split") != "test":
    raise ValueError(f"Question {second_case_id} is not confirmed in the held-out test split.")

release_match = re.search(r"(?<!\d)(\d{2}\.\d{2})(?!\d)", second_row.get("Tags", ""))
second_release = release_match.group(1) if release_match else None
if second_release != "22.04":
    raise ValueError("The selected kernel-removal case must have Ubuntu 22.04 recorded in its Ask Ubuntu tags.")

additional_cases = [{
    "case_id": second_case_id,
    "case_type": "Held-out Ask Ubuntu",
    "decision_path": "APT plan removes a Linux kernel",
    "expected_status": "needs_review",
    "title": second_row["Title"],
    "body": second_row["BodyText"],
    "product_type": None,
    "ubuntu_release": second_release,
    "exclude_question_ids": [second_case_id],
}, {
    "case_id": "SYN-DNS-001",
    "case_type": "Synthetic demonstration",
    "decision_path": "Read-only DNS recommendation",
    "expected_status": "recommendation",
    "title": "Ubuntu Server 22.04 cannot resolve www.google.com",
    "body": (
        "On Ubuntu Server 22.04, lookups for www.google.com time out even though "
        "the server has network connectivity by IP. I have not changed the DNS "
        "configuration. I want a read-only check of whether this hostname resolves; "
        "do not change DNS settings."
    ),
    "product_type": "ubuntu_server",
    "ubuntu_release": "22.04",
    "exclude_question_ids": [],
}, {
    "case_id": "SYN-APT-MISSING-001",
    "case_type": "Synthetic demonstration",
    "decision_path": "Missing package-repair details gate",
    "expected_status": "needs_review",
    "title": "APT upgrade has unmet dependencies",
    "body": "I need help fixing it.",
    "product_type": None,
    "ubuntu_release": None,
    "exclude_question_ids": [],
}, {
    "case_id": "SYN-OUT-OF-SCOPE-001",
    "case_type": "Synthetic demonstration",
    "decision_path": "Abstain outside Ubuntu support",
    "expected_status": "unsupported",
    "title": "Windows Server domain controller",
    "body": "Active Directory replication failed.",
    "product_type": None,
    "ubuntu_release": None,
    "exclude_question_ids": [],
}]

completed_ids = {item["case_id"] for item in current_run_results}

ADDITIONAL_CASES_ATTEMPTED = True
for case in additional_cases:
    # Inspect the individual evidence-producing components before integration.
    if case["case_id"] in {"1503371", "SYN-DNS-001"}:
        import pandas as pd
        from pprint import pprint
        from src.models.classification import predict_ticket
        from src.support_agent import _official_query

        print("\n=== Standalone component results:", case["case_id"], "===")
        m3_standalone = predict_ticket(
            classifier_artifact, case["title"], case["body"]
        )
        print("M3 classification:", m3_standalone)
        print("M3 saved review threshold:",
              classifier_artifact["review_policy"]["threshold"])

        m4_standalone = retriever.retrieve(
            case["title"], case["body"], k=5,
            exclude_question_ids=case["exclude_question_ids"],
        )
        print("M4 method / weak evidence / top score / margin:",
              m4_standalone["method"], m4_standalone["weak_evidence"],
              m4_standalone["top_score"], m4_standalone["relative_margin"])
        community_rows = []
        for hit in m4_standalone["evidence"]:
            community_rows.append({
                "rank": hit.get("rank"),
                "question_id": hit.get("question_id"),
                "answer_id": hit.get("answer_id"),
                "title": hit.get("title"),
                "score": hit.get("score"),
                "question_url": (hit.get("sources", {}).get("question") or {}).get("source_url"),
                "answer_url": (hit.get("sources", {}).get("answer") or {}).get("source_url"),
                "answer_excerpt": (hit.get("answer_text") or "")[:500],
            })
        display(pd.DataFrame(community_rows))

        docs_query = _official_query(case["title"], case["body"])
        docs_standalone = official_retriever.search(
            docs_query,
            k=len(official_retriever.chunks),
            product_type=case["product_type"],
            ubuntu_release=case["ubuntu_release"],
            deduplicate_pages=True,
        )
        print("Official-docs query:", docs_query)
        print("Standalone official-docs results (top 10 pages):")
        docs_rows = [{
            "rank": hit.get("rank"),
            "title": hit.get("title"),
            "score": hit.get("score"),
            "scope_status": hit.get("scope_status"),
            "release_scope_status": hit.get("release_scope_status"),
            "product_scope_status": hit.get("product_scope_status"),
            "url": hit.get("url"),
            "excerpt": (hit.get("text") or "")[:400],
        } for hit in docs_standalone[:10]]
        display(pd.DataFrame(docs_rows))

    if case["case_id"] in completed_ids:
        print("Already completed; skipping repeat generation:", case["case_id"])
        continue

    case_prepared = agent.prepare(
        case["title"],
        case["body"],
        exclude_question_ids=case["exclude_question_ids"],
        product_type=case["product_type"],
        ubuntu_release=case["ubuntu_release"],
    )

    print("Integrated prepare() classification:", case_prepared["classification"])
    print("Integrated retrieval lane summary:")
    pprint(case_prepared["retrieval_lanes"])
    print("Sources selected for the integrated prompt:")
    integrated_sources = [{
        "source_id": source.get("source_id"),
        "lane": source.get("lane"),
        "title": source.get("title"),
        "score": source.get("score"),
        "scope_status": source.get("scope_status"),
        "url": source.get("url"),
    } for source in case_prepared.get("sources", [])]
    display(pd.DataFrame(integrated_sources))

    # Confirm that each selected input actually reaches the decision path it is meant to demonstrate.
    if case["case_id"] == "1503371":
        assert case_prepared["apt_transaction"].get("kernel_removal_planned"), "Selected APT plan no longer shows kernel removal."
        assert not case_prepared["classification"].get("human_review"), "Kernel case should show that safety escalation is independent of M3 review."
        assert not case_prepared["out_of_scope"] and not case_prepared["missing_information"]
        assert case_prepared.get("sources"), "Kernel case needs retrieved evidence to reach the model before the deterministic safety override."
    elif case["case_id"] == "SYN-DNS-001":
        assert not case_prepared["out_of_scope"] and not case_prepared["missing_information"]
        assert not case_prepared["classification"].get("human_review"), "DNS case unexpectedly triggers the M3 review flag."
        documented_query = any(
            source.get("lane") == "official_ubuntu_docs"
            and source.get("scope_status") == "match"
            and "resolvectl" in source.get("title", "").casefold()
            and "query HOSTNAME" in (source.get("answer_excerpt") or "")
            for source in case_prepared.get("sources", [])
        )
        assert documented_query, "No release-matched official source documents the read-only resolvectl query."
    elif case["case_id"] == "SYN-APT-MISSING-001":
        assert not case_prepared["out_of_scope"]
        assert "Ubuntu version" in case_prepared["missing_information"]
        assert "complete package-manager error message" in case_prepared["missing_information"]
    elif case["case_id"] == "SYN-OUT-OF-SCOPE-001":
        assert case_prepared["out_of_scope"], "Selected Windows request was not detected as out of scope."

    case_result = None
    case_result = agent.generate(case_prepared)
    current_run_results.append({
        "case_id": case["case_id"],
        "case_type": case["case_type"],
        "decision_path": case["decision_path"],
        "expected_status": case["expected_status"],
        "prepared": case_prepared,
        "result": case_result,
    })
    completed_ids.add(case["case_id"])
    print("\n---", case["case_id"], "|", case["decision_path"], "---")
    print("Expected / actual status:", case["expected_status"], "/", case_result.get("status"))
    print("M3 category / confidence / review:", case_prepared.get("category"),
          case_prepared["classification"].get("confidence"),
          case_prepared["classification"].get("human_review"))
    print("Missing information:", case_prepared.get("missing_information"))
    print("Out of scope:", case_prepared.get("out_of_scope"))
    print("Sources prepared:", len(case_prepared.get("sources", [])))
    print("Gemini/model usage:", case_result.get("usage", "not called or unavailable"))
    print("Answer:", case_result.get("answer") or "(no answer)")
    print("Returned / withheld steps:", len(case_result.get("steps", [])),
          "/", len(case_result.get("withheld_steps", [])))
    print("Answer validation:", case_result.get("answer_validation"))
    print("Step validation:", case_result.get("step_validation"))
    print("Safety decision:", (case_result.get("safety_assessment") or {}).get("decision"))
    print("Review reasons:", case_result.get("review_reasons"))
    print("Latency seconds:", case_result.get("seconds"))

assert completed_ids == set(expected_ids), "A selected scenario is missing from this run."
print("\nCase results available:", sorted(completed_ids))
print("\nDetailed current-code outcomes:")
for item in current_run_results:
    case_result = item["result"]
    print("\n###", item["case_id"], "|", item["decision_path"])
    print("Status:", case_result.get("status"))
    print("Summary:", case_result.get("summary"))
    print("Answer:", case_result.get("answer") or "(none)")
    print("Answer citations:", case_result.get("answer_citations"))
    print("Steps:", case_result.get("steps"))
    print("Withheld steps and reasons:", case_result.get("withheld_steps"))
    print("Missing information:", case_result.get("missing_information"))
    print("Review reasons:", case_result.get("review_reasons"))
    print("Step validation:", case_result.get("step_validation"))
    print("Safety:", case_result.get("safety_assessment"))



=== Standalone component results: 1503371 ===
M3 classification: {'category': 'Software & Packages', 'confidence': 0.9880989759340439, 'human_review': False}
M3 saved review threshold: 0.7492341336823973
M4 method / weak evidence / top score / margin: dense False 0.8687826991081238 0.013514565797792293


,rank,question_id,answer_id,title,score,question_url,answer_url,answer_excerpt
0,1,1278976,1278988,dpkg error removing linux-image-4.4.0-190-gene...,0.868783,https://askubuntu.com/questions/1278976,https://askubuntu.com/a/1278988,You have suspicious binaries in the `/etc/grub...
1,2,20824,20883,Error while removing the new kernel 2.6.37,0.857041,https://askubuntu.com/questions/20824,https://askubuntu.com/a/20883,I found the fix for this bug by this codes:\n\...
2,3,688609,690607,apt-get - cannot install or remove any pacakge...,0.838015,https://askubuntu.com/questions/688609,https://askubuntu.com/a/690607,You have a syntax error in\n\n\n\n```\n/etc/de...
3,4,419095,419132,Unable to remove old kernels,0.787537,https://askubuntu.com/questions/419095,https://askubuntu.com/a/419132,"To prevent future 'no space in boot/' errors, ..."
4,5,971833,971844,Trying to free disk space on production server,0.708612,https://askubuntu.com/questions/971833,https://askubuntu.com/a/971844,First install the following package:\n\n\n\n`s...


Official-docs query: How to fix fix broken install command (due to zoom installation)
Installing zoom-meeting created a broken package. Since then I've tried everything, but nothing can be done.


[Image: Package Manager screenshot] (https://i.stack.imgur.com/whIG8.png) (https://i.stack.imgur.com/whIG8.png)


When I enter the command `sudo apt-get --purge remove zoom` I get these results:


```
Reading package lists... Done
Build dependency tree... Done
Read status information... Done      
The following package has been installed automatically and is no longer required:
  libwmf0.2-7
Please use "sudo apt autoremove" to remove it.
The following packages will be REMOVED:
  linux-image-6.5.0-14-generic zoom*
0 updated, 0 newly installed, 2 to be removed and 26 not updated.
3 partially installed or removed.
After this operation, 698 MB of disk space will be freed.
Would you like to continue? [Y/n] o
(Reading the database... 533735 files and directories already installed)
Remove linux-imag

,rank,title,score,scope_status,release_scope_status,product_scope_status,url,excerpt
0,1,Install and manage packages,0.486557,unknown,unknown,unknown,https://ubuntu.com/server/docs/how-to/software...,requires fix\nH : Half-installed - removal fai...
1,2,Ubuntu 26.04 LTS NVIDIA driver guidance,0.431852,mismatch,mismatch,unknown,https://documentation.ubuntu.com/release-notes...,Redo changes:\n$ sudo apt history-redo 0\ncows...
2,3,Ubuntu 22.04 manpage: apt-get,0.428330,unknown,match,unknown,https://manpages.ubuntu.com/manpages/jammy/man...,can omit any packages to\npermit APT to deduce...
3,4,Ubuntu manpage: df,0.212032,mismatch,mismatch,unknown,https://manpages.ubuntu.com/manpages/resolute/...,Ubuntu Manpage: df - report file system space ...
4,5,Install Ubuntu Desktop,0.200196,mismatch,mismatch,unknown,https://ubuntu.com/desktop/docs/en/26.04/tutor...,Ctrl+Alt+T to bring up a Terminal window.\nTyp...
5,6,Install or remove software,0.194892,mismatch,mismatch,unknown,https://ubuntu.com/desktop/docs/en/26.04/how-t...,Install or remove software - Ubuntu Desktop do...
6,7,Ubuntu manpage: ls,0.184254,mismatch,mismatch,unknown,https://manpages.ubuntu.com/manpages/resolute/...,[-f ] [-d|--directory]\n[-h|--human-readable] ...
7,8,Ubuntu manpage: lsblk,0.170718,mismatch,mismatch,unknown,https://manpages.ubuntu.com/manpages/resolute/...,the kernel build.\nAUTHORS\nMilan Broz <gmazyl...
8,9,Install NVIDIA drivers,0.165694,unknown,unknown,unknown,https://ubuntu.com/desktop/docs/en/latest/how-...,apt show nvidia-driver-VERSION | grep ^Support...
9,10,Configuring networks and name resolution,0.155435,unknown,unknown,unknown,https://ubuntu.com/server/docs/explanation/net...,backends\nThe rid idmap backend\nThe autorid i...


Integrated prepare() classification: {'category': 'Software & Packages', 'confidence': 0.9880989759340439, 'human_review': False}
Integrated retrieval lane summary:
{'askubuntu': {'error': None,
               'method': 'dense',
               'results': ['AU-1278976',
                           'AU-20824',
                           'AU-688609',
                           'AU-419095',
                           'AU-971833'],
               'status': 'ok',
               'weak_evidence': False},
 'official_ubuntu_docs': {'category_preference_applied': True,
                          'dropped_low_relevance': 5,
                          'error': None,
                          'excluded_scope_mismatch': ['OD-2',
                                                      'OD-4',
                                                      'OD-5',
                                                      'OD-6',
                                                      'OD-7',
                               

,source_id,lane,title,score,scope_status,url
0,OD-3,official_ubuntu_docs,Ubuntu 22.04 manpage: apt-get,0.428330,match,https://manpages.ubuntu.com/manpages/jammy/man...
1,OD-1,official_ubuntu_docs,Install and manage packages,0.486557,unknown,https://ubuntu.com/server/docs/how-to/software...
2,AU-1278976,askubuntu,dpkg error removing linux-image-4.4.0-190-gene...,0.868783,unknown,https://askubuntu.com/a/1278988
3,AU-20824,askubuntu,Error while removing the new kernel 2.6.37,0.857041,unknown,https://askubuntu.com/a/20883
4,AU-688609,askubuntu,apt-get - cannot install or remove any pacakge...,0.838015,unknown,https://askubuntu.com/a/690607
5,AU-419095,askubuntu,Unable to remove old kernels,0.787537,unknown,https://askubuntu.com/a/419132
6,AU-971833,askubuntu,Trying to free disk space on production server,0.708612,unknown,https://askubuntu.com/a/971844


... (8 outputs left out)

,source_id,lane,title,score,scope_status,url
0,OD-1,official_ubuntu_docs,Ubuntu 22.04 manpage: apt-get,0.429070,unknown,https://manpages.ubuntu.com/manpages/jammy/man...
1,OD-2,official_ubuntu_docs,Install and manage packages,0.377710,unknown,https://ubuntu.com/server/docs/how-to/software...
2,OD-3,official_ubuntu_docs,Install Ubuntu Desktop,0.290421,unknown,https://ubuntu.com/desktop/docs/en/26.04/tutor...
3,OD-4,official_ubuntu_docs,Ubuntu 26.04 LTS NVIDIA driver guidance,0.274541,unknown,https://documentation.ubuntu.com/release-notes...
4,AU-1452233,askubuntu,Unable to resolve unmet dependencies,0.755698,unknown,https://askubuntu.com/a/1452235
5,AU-245253,askubuntu,"""Unmet Dependencies"" problem when trying apt-g...",0.707613,unknown,https://askubuntu.com/a/245276
6,AU-1405425,askubuntu,can't install updates and got the package syst...,0.644783,unknown,https://askubuntu.com/a/1405427
7,AU-564282,askubuntu,"apt-get, unmet dependencies, ... ""but it is no...",0.629395,unknown,https://askubuntu.com/a/606896
8,AU-127727,askubuntu,Broken dependencies due to libjackd,0.613898,unknown,https://askubuntu.com/a/129333



--- SYN-APT-MISSING-001 | Missing package-repair details gate ---
Expected / actual status: needs_review / needs_review
M3 category / confidence / review: Software & Packages 0.9988587146547616 False
Missing information: ['Ubuntu version', 'complete package-manager error message']
Out of scope: False
Sources prepared: 9
Gemini/model usage: not called or unavailable
Answer: (no answer)
Returned / withheld steps: 0 / 0
Answer validation: {'status': 'not_checked', 'issues': []}
Step validation: {'status': 'not_checked', 'issues': []}
Safety decision: not_evaluated
Review reasons: ['Package repair depends on the Ubuntu release and complete error output.']
Latency seconds: None
Integrated prepare() classification: {'category': 'Network & Connectivity', 'confidence': 0.910868522426446, 'human_review': False}
Integrated retrieval lane summary:
{'askubuntu': {'error': None,
               'method': 'dense',
               'results': ['AU-1246013',
                           'AU-32261',
      

,source_id,lane,title,score,scope_status,url
0,OD-2,official_ubuntu_docs,Networking,0.217656,unknown,https://ubuntu.com/server/docs/how-to/networking/
1,OD-3,official_ubuntu_docs,Configuring networks and name resolution,0.211380,unknown,https://ubuntu.com/server/docs/explanation/net...
2,OD-1,official_ubuntu_docs,Ubuntu 26.04 LTS NVIDIA driver guidance,0.231564,unknown,https://documentation.ubuntu.com/release-notes...
3,OD-4,official_ubuntu_docs,Install and manage packages,0.200734,unknown,https://ubuntu.com/server/docs/how-to/software...
4,AU-1246013,askubuntu,can't join ubuntu server 20.04 to active direc...,0.494568,unknown,https://askubuntu.com/a/1246185
5,AU-32261,askubuntu,How can I add Ubuntu to a windows domain?,0.410095,unknown,https://askubuntu.com/a/60674
6,AU-295814,askubuntu,Ubuntu joining a active directory domain,0.381511,unknown,https://askubuntu.com/a/295849
7,AU-717840,askubuntu,Samba authentication fails,0.319367,unknown,https://askubuntu.com/a/1021041
8,AU-1471014,askubuntu,Ubuntu 22.04 can't accept RDP connecting from ...,0.313471,unknown,https://askubuntu.com/a/1484271



--- SYN-OUT-OF-SCOPE-001 | Abstain outside Ubuntu support ---
Expected / actual status: unsupported / unsupported
M3 category / confidence / review: Network & Connectivity 0.910868522426446 False
Missing information: []
Out of scope: True
Sources prepared: 9
Gemini/model usage: not called or unavailable
Answer: (no answer)
Returned / withheld steps: 0 / 0
Answer validation: {'status': 'not_checked', 'issues': []}
Step validation: {'status': 'not_checked', 'issues': []}
Safety decision: not_evaluated
Review reasons: ['Route the request to the team that supports the reported platform.']
Latency seconds: None

Case results available: ['1503371', 'SYN-APT-MISSING-001', 'SYN-DNS-001', 'SYN-OUT-OF-SCOPE-001']

Detailed current-code outcomes:

### 1503371 | APT plan removes a Linux kernel
Status: needs_review
Summary: The explanation was withheld because its evidence could not be verified.
Answer: (none)
Answer citations: []
Steps: []
Withheld steps and reasons: []
Missing information: ['Con

## 4. Compare Expected and Actual Decisions

The table compares each selected decision path with the status returned by the current run. A mismatch should be reviewed against the answer and review reasons.

- **M3 confidence and review flag** describe the classifier, not answer quality.
- **Agent status** is `recommendation`, `needs_review`, or `unsupported`. A recommendation passed the implemented checks without a case-specific review reason; it is not a guarantee of correctness.
- **Safety decision** is separate from agent status. `escalate` means a safety rule triggered; `allow` means the implemented checks did not trigger escalation; `not_evaluated` means they did not run on that path.
- **Answer and step validation** report only the implemented evidence, citation, scope, and safety checks. They do not establish technical correctness or execute commands.

`engineer_review_required` is true for every result in the current implementation, so it does not distinguish the cases. It also does not create an automatic handoff to an engineer.

## 5. Decision Behavior in the Current Implementation

`prepare()` classifies the ticket and gathers evidence. `generate()` first applies the out-of-scope, missing-repair-details, and no-source gates. If a case passes those gates, the model drafts a response; deterministic checks then validate citations and steps and may add review reasons or withhold unsafe suggestions.

A planned Linux-kernel removal forces `needs_review` and a safety escalation regardless of the model's wording. A missing Ubuntu release or incomplete APT error log stops package-repair generation and identifies what is needed. An out-of-scope platform returns `unsupported` without a generated answer.

These four cases demonstrate selected paths, not every code branch or overall system quality. The missing-information and out-of-scope paths are synthetic demonstrations. Broader quality measurement and improvements based on evaluation results are reserved for M8.

## 6. Results

The table below is generated from the four current-code runs. It distinguishes the held-out ticket from synthetic demonstrations and shows both the intended path and the returned status.

In [4]:
import pandas as pd
from IPython.display import display

comparison_rows = []
for item in current_run_results:
    prepared_case = item["prepared"]
    case_result = item["result"]
    comparison_rows.append({
        "Case ID": item["case_id"],
        "Case type": item["case_type"],
        "Decision path being exercised": item["decision_path"],
        "Expected agent status": item["expected_status"],
        "Data split": prepared_case.get("ticket", {}).get("split", "synthetic" if item["case_type"] == "Synthetic demonstration" else "test"),
        "M3 predicted category": prepared_case.get("category"),
        "M3 classifier confidence": prepared_case.get("classification", {}).get("confidence"),
        "M3 classifier review flag": prepared_case.get("classification", {}).get("human_review"),
        "Agent status": case_result.get("status"),
        "Answer validation": (case_result.get("answer_validation") or {}).get("status"),
        "Step validation": (case_result.get("step_validation") or {}).get("status"),
        "Returned steps": len(case_result.get("steps") or []),
        "Withheld steps": len(case_result.get("withheld_steps") or []),
        "Safety decision": (case_result.get("safety_assessment") or {}).get("decision"),
        "Case-specific review reasons": case_result.get("review_reasons") or [],
    })

assert [row["Case ID"] for row in comparison_rows] == ["1503371", "SYN-DNS-001", "SYN-APT-MISSING-001", "SYN-OUT-OF-SCOPE-001"]
display(pd.DataFrame(comparison_rows))

,Case ID,Case type,Decision path being exercised,Expected agent status,Data split,M3 predicted category,M3 classifier confidence,M3 classifier review flag,Agent status,Answer validation,Step validation,Returned steps,Withheld steps,Safety decision,Case-specific review reasons
0,1503371,Held-out Ask Ubuntu,APT plan removes a Linux kernel,needs_review,test,Software & Packages,0.988099,False,needs_review,failed,no_steps,0,0,escalate,[Community answers indicate a syntax error in ...
1,SYN-DNS-001,Synthetic demonstration,Read-only DNS recommendation,recommendation,synthetic,Network & Connectivity,0.997987,False,recommendation,passed,passed,1,0,allow,[]
2,SYN-APT-MISSING-001,Synthetic demonstration,Missing package-repair details gate,needs_review,synthetic,Software & Packages,0.998859,False,needs_review,not_checked,not_checked,0,0,not_evaluated,[Package repair depends on the Ubuntu release ...
3,SYN-OUT-OF-SCOPE-001,Synthetic demonstration,Abstain outside Ubuntu support,unsupported,synthetic,Network & Connectivity,0.910869,False,unsupported,not_checked,not_checked,0,0,not_evaluated,[Route the request to the team that supports t...
